# 03 · Forecast backtest (Phase 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arooj-zehra/pakistan-water-watch/blob/main/notebooks/03_backtest.ipynb)

How good are the 1 to 10 day inflow forecasts, judged only on years the model never saw?

* **Model:** LightGBM quantile regression per horizon, both dams pooled, predicting the change in log inflow (so "no change" is the persistence forecast).
* **Folds:** for each test year (2021, 2023, 2024, 2025) the model trains on earlier years, calibrates its intervals on the year before, and is scored on the test year. 2026 so far is the final holdout.
* **Baselines:** persistence (tomorrow equals today), climatology (median for that day of year), seasonal naive (same day last year).

Before opening this, run on your computer and push: `python -m models.backtest` (it writes `models/results/`, `docs/img/` and `docs/MODEL_CARD.md`).

Then in Colab: **Runtime → Run all**. Set `RERUN = True` below to retrain everything inside Colab instead (about 5 minutes).

## Setup
Clones the repo, or if this session already has a copy, resets it to the latest version on GitHub (anything a previous run changed inside Colab is discarded), then installs the requirements. Takes about a minute the first time.

In [ ]:
import os, sys

REPO_URL = "https://github.com/arooj-zehra/pakistan-water-watch.git"
REPO_DIR = "/content/pakistan-water-watch"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if os.path.isdir(REPO_DIR):
        # The Colab copy is throwaway: earlier runs rewrite data files inside it,
        # which would make a plain `git pull` refuse. Reset it to GitHub instead,
        # and delete untracked/ignored files too (incl. the old data/pww.db) so
        # every run rebuilds the database from scratch.
        !git -C {REPO_DIR} fetch -q origin
        !git -C {REPO_DIR} reset -q --hard origin/main
        !git -C {REPO_DIR} clean -q -fdx
    else:
        !git clone -q {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # running locally: work from the repo root

sys.path.insert(0, os.getcwd())
!git log --oneline -1
print("Working in", os.getcwd())

## Results
Reads the committed results, or retrains first if `RERUN = True` or nothing has been committed yet.

In [ ]:
import json
from pathlib import Path
import io
import pandas as pd
from IPython.display import Image, display

def show(fig):
    """Render a matplotlib Figure as a PNG (works in Colab and locally, no pyplot needed)."""
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    display(Image(data=buf.getvalue()))

RERUN = False
summary_path = Path("models/results/backtest_summary.json")
if RERUN or not summary_path.exists():
    !python -m models.backtest
res = json.loads(summary_path.read_text())
live = res["summary"]["live"]
print("Data through", res["data_through"], "| results generated", res["generated_at"])

rows = []
for h, d in live.items():
    s, ho = d["all_test_years"], d["holdout"] or {}
    rows.append({"horizon (days)": int(h), "rows": s["n"],
                 "MAE model": s["mae_model"], "MAE persistence": s["mae_persistence"],
                 "MAE climatology": s["mae_climatology"], "MAE seasonal naive": s["mae_seasonal_naive"],
                 "NSE": s["nse_model"], "skill vs persistence": s["skill_vs_persistence"],
                 "90% coverage": s["coverage90"],
                 "holdout skill": ho.get("skill_vs_persistence"), "holdout 90% coverage": ho.get("coverage90")})
table = pd.DataFrame(rows).set_index("horizon (days)")
table.style.format({c: "{:,.0f}" for c in table.columns if c.startswith("MAE")} |
                   {"NSE": "{:.3f}"} | {c: "{:.0%}" for c in ["skill vs persistence", "90% coverage", "holdout skill", "holdout 90% coverage"]})

**Done-when check** from the plan: the model beats persistence at 3+ day horizons, and calibrated 90% intervals hold 85–95% of outcomes on the holdout.

In [ ]:
beats = all(live[h]["all_test_years"]["skill_vs_persistence"] > 0 for h in live if int(h) >= 3)
cov = [live[h]["holdout"]["coverage90"] for h in live if live[h]["holdout"]]
print("Beats persistence at 3+ days:", "YES" if beats else "NO")
print("Holdout 90% coverage:", ", ".join(f"{c:.0%}" for c in cov), "->", "YES" if all(0.85 <= c <= 0.95 for c in cov) else "NO")

## Skill and calibration

In [ ]:
display(Image("docs/img/skill_by_horizon.png"))
display(Image("docs/img/reliability.png"))

## By season
The melt season (April to September) carries most of the water and most of the error.

In [ ]:
pd.DataFrame([{"horizon": int(h), "season": k, "MAE model": s["mae_model"],
               "skill vs persistence": s["skill_vs_persistence"], "90% coverage": s["coverage90"]}
              for h, d in live.items() for k, s in d["by_season"].items()]
            ).pivot(index="horizon", columns="season").style.format("{:.2f}")

## The look-ahead trap
The live model only knows weather up to the day before the forecast. The oracle variant is handed the real weather of the forecast window, which nobody can know. The gap is the most a perfect weather forecast could add.

In [ ]:
orc = res["summary"].get("oracle", {})
pd.DataFrame({"live skill": {int(h): d["all_test_years"]["skill_vs_persistence"] for h, d in live.items()},
              "perfect-weather skill": {int(h): d["all_test_years"]["skill_vs_persistence"] for h, d in orc.items()}}
            ).style.format("{:.0%}")

## One test year up close
Forecasts made 5 days ahead against what actually came in, with the 90% interval. Change `STATION`, `YEAR` or `H` to look at others.

In [ ]:
from matplotlib.figure import Figure
STATION, YEAR, H = "tarbela", 2023, 5
p = pd.read_csv("models/results/backtest_predictions.csv", parse_dates=["issue_date", "target_date"])
p = p[(p.variant == "live") & (p.sid == STATION) & (p.year == YEAR) & (p.horizon == H)].sort_values("target_date")
fig = Figure(figsize=(11, 4)); ax = fig.subplots()
ax.fill_between(p.target_date, p.lo90, p.hi90, color="#3e8e98", alpha=0.25, lw=0, label="90% interval")
ax.plot(p.target_date, p.model, color="#3e8e98", lw=1.4, label=f"forecast made {H} days earlier")
ax.plot(p.target_date, p.persistence, color="#b5a27f", lw=0.9, ls="--", label="persistence")
ax.scatter(p.target_date, p.q_target, s=6, color="#1c2b33", zorder=3, label="actual inflow")
ax.set_ylabel("cusecs"); ax.yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}"); ax.set_title(f"{STATION.title()} {YEAR}, {H}-day forecasts"); ax.legend(frameon=False, fontsize=8)
for s in ("top", "right"): ax.spines[s].set_visible(False)
show(fig)

## What drives the forecasts
Mean absolute SHAP contribution of each feature to the production median model (LightGBM's built-in `pred_contrib`).

In [ ]:
art = Path("models/artifacts") / Path("models/artifacts/LATEST").read_text().strip()
meta = json.loads((art / "meta.json").read_text())
fig = Figure(figsize=(11, 3.8)); axes = fig.subplots(1, 2)
for ax, h in zip(axes, [min(meta["importance"], key=int), max(meta["importance"], key=int)]):
    imp = pd.Series(meta["importance"][h]).sort_values()
    ax.barh(imp.index, imp.values, color="#3e8e98"); ax.set_title(f"{h}-day horizon")
    for s in ("top", "right"): ax.spines[s].set_visible(False)
fig.tight_layout(); show(fig)

## Today's forecast
Written by `python -m models.predict` (and by the daily job from Phase 4).

In [ ]:
fpath = Path("web/public/data/forecast.json")
if not fpath.exists():
    print("No forecast.json yet: run  python -m models.predict  on your computer and push.")
else:
    fc = json.loads(fpath.read_text())
    print("Model", fc["model_version"], "issued", fc["issued_at"])
    for sid, s in fc["stations"].items():
        print(f"\n{sid.title()}: {s['inflow_today']:,} cusecs on {s['issue_date']}")
        for w in s["warnings"]: print("  !", w)
        display(pd.DataFrame(s["forecasts"])[["h", "target_date", "p05", "p25", "p50", "p75", "p95", "explanation"]])